# 04 Evaluation and Report-Ready Results

This notebook loads final model-comparison outputs and creates report-ready tables/figures.

It focuses on:
- Four-model comparison.
- Threshold evaluation.
- Confusion matrix.
- Top-K evaluation.
- Interpretation for BAB IV.

In [ ]:
from pathlib import Path
import sys

# Works whether the notebook is launched from repository root or from notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


import pandas as pd
import matplotlib.pyplot as plt

## 1. Load final evaluation artifacts

In [ ]:
def read_first_existing(paths):
    for path in paths:
        if path.exists():
            print("Loaded:", path)
            return pd.read_csv(path)
    raise FileNotFoundError("None of these files exist:\n" + "\n".join(str(p) for p in paths))


model_comparison = read_first_existing([
    TABLES_DIR / "four_model_comparison_temporal_graph_final.csv",
    TABLES_DIR / "four_model_comparison_temporal_graph.csv",
])

threshold_eval = read_first_existing([
    TABLES_DIR / "final_threshold_evaluation_four_model_best.csv",
    TABLES_DIR / "final_threshold_evaluation.csv",
])

topk_eval = read_first_existing([
    TABLES_DIR / "top_k_evaluation_four_model_best.csv",
    TABLES_DIR / "top_k_evaluation.csv",
])

confusion_matrix_df = read_first_existing([
    TABLES_DIR / "confusion_matrix_tuned_four_model_best.csv",
    TABLES_DIR / "confusion_matrix_tuned.csv",
])

display(model_comparison)
display(threshold_eval)
display(topk_eval)
display(confusion_matrix_df)

## 2. Four-model comparison figure

In [ ]:
plot_df = model_comparison.sort_values("val_pr_auc", ascending=True).copy()

ax = plot_df.plot(
    kind="barh",
    x="model",
    y="val_pr_auc",
    legend=False,
    figsize=(8, 4),
)
ax.set_title("Four-Model Comparison by Validation PR-AUC")
ax.set_xlabel("Validation PR-AUC / Average Precision")
ax.set_ylabel("Model")
plt.tight_layout()

out_path = FIGURES_DIR / "four_model_validation_pr_auc.png"
plt.savefig(out_path, dpi=150)
plt.show()

print("Saved:", out_path)

## 3. Test metric comparison figure

In [ ]:
metric_cols = ["test_pr_auc", "test_f1", "test_precision", "test_recall"]
plot_df = model_comparison.set_index("model")[metric_cols].sort_values("test_pr_auc", ascending=False)

ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_title("Test Metrics by Model")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

out_path = FIGURES_DIR / "four_model_test_metrics.png"
plt.savefig(out_path, dpi=150)
plt.show()

print("Saved:", out_path)

## 4. Threshold comparison

In [ ]:
display(threshold_eval)

ax = threshold_eval.plot(
    kind="bar",
    x="setting",
    y=["f1", "precision", "recall"],
    figsize=(8, 4),
)
ax.set_title("Threshold Evaluation")
ax.set_xlabel("Setting")
ax.set_ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

out_path = FIGURES_DIR / "threshold_evaluation.png"
plt.savefig(out_path, dpi=150)
plt.show()

print("Saved:", out_path)

## 5. Top-K evaluation

In [ ]:
display(topk_eval)

ax = topk_eval.plot(
    kind="line",
    x="k",
    y=["precision_at_k", "recall_at_k"],
    marker="o",
    figsize=(8, 4),
)
ax.set_title("Top-K Evaluation")
ax.set_xlabel("K highest-risk transactions reviewed")
ax.set_ylabel("Score")
plt.tight_layout()

out_path = FIGURES_DIR / "top_k_evaluation.png"
plt.savefig(out_path, dpi=150)
plt.show()

print("Saved:", out_path)

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix_df.copy()

# Handle both CSV styles: with unnamed index column or with explicit label column.
if "Unnamed: 0" in cm.columns:
    cm = cm.set_index("Unnamed: 0")

display(cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm.values)
plt.title("Tuned Confusion Matrix")
plt.xticks(range(len(cm.columns)), cm.columns, rotation=30, ha="right")
plt.yticks(range(len(cm.index)), cm.index)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, f"{int(cm.values[i, j]):,}", ha="center", va="center")

plt.tight_layout()
out_path = FIGURES_DIR / "confusion_matrix_tuned.png"
plt.savefig(out_path, dpi=150)
plt.show()

print("Saved:", out_path)

## 7. Final interpretation summary

In [ ]:
best = model_comparison.sort_values("val_pr_auc", ascending=False).iloc[0]

summary_lines = [
    f"Best selected model: {best['model']}",
    f"Feature set: {best['feature_set']}",
    f"Validation PR-AUC: {best['val_pr_auc']:.6f}",
    f"Test PR-AUC: {best['test_pr_auc']:.6f}",
    f"Test F1: {best['test_f1']:.6f}",
    f"Test precision: {best['test_precision']:.6f}",
    f"Test recall: {best['test_recall']:.6f}",
]

for line in summary_lines:
    print(line)

summary_path = TABLES_DIR / "final_result_summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print("Saved:", summary_path)

## BAB IV interpretation draft

Use this interpretation in the report:

- Four algorithms were compared: Logistic Regression, Random Forest, XGBoost, and LightGBM.
- The best model was selected using validation PR-AUC because AML detection is a rare-event ranking problem.
- LightGBM is selected as the final model because it provides the strongest validation PR-AUC and the best balance of precision, F1, and alert volume.
- Logistic Regression may show high recall or high test PR-AUC, but it produces too many alerts and has weak validation performance.
- Threshold evaluation should be interpreted as operational modes:
  - lower/default threshold: better recall and F1,
  - stricter threshold: higher precision and fewer alerts.
- Top-K evaluation is important because AML analysts usually review a limited number of highest-risk transactions.